# 🚀 ComfyUI + LTX 2.3 GGUF — Multi-Account Video Generator

**Dựa trên:** pogscafe/comfyui-kaggle-march-2025 (⭐2202) + Lightricks/ComfyUI-LTXVideo (⭐3.9k)

**Cơ chế:** Notebook start → ComfyUI → Tunnel Pinggy → Ghi URL lên GitHub Gist
→ Hermes tự động phát hiện → Dispatch job gen video

**Model:** LTX-2.3 22B GGUF Q2_K (12.4 GB) — fits T4 16GB ✅

---


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
BACKEND_NAME = "kaggle-a"
MODEL_QUANT = "Q2_K"

# Gist ID cố định — Hermes dùng chung Gist này để biết URL
GIST_ID = "8da27f2e6e0d8809a043712cd90f9237"

print(f"Config: {BACKEND_NAME} | {MODEL_QUANT}")

---
## 1. Cài môi trường

In [ ]:
%%time
import os, sys, subprocess, shutil, threading, time, requests, json
from datetime import datetime
from IPython.display import display, HTML, clear_output

home_dir = '/kaggle/working'
COMFY_DIR = f'{home_dir}/ComfyUI'
VENV_DIR = f'{home_dir}/venv'
os.chdir(home_dir)

# Fix venv Python 3.12: dùng virtualenv thay vì venv
!pip install -q virtualenv

if not os.path.exists(VENV_DIR):
    subprocess.run(f'virtualenv {VENV_DIR} -p $(which python3.10)', shell=True, check=False)

if os.path.exists(f'{VENV_DIR}/bin/python'):
    python = f'{VENV_DIR}/bin/python'
    pip = f'{VENV_DIR}/bin/pip'
    USE_VENV = True
else:
    python = sys.executable
    pip = 'pip3 install --break-system-packages'
    USE_VENV = False

print(f'Python: {python}')
print(f'USe venv: {USE_VENV}')

In [ ]:
%%time
# Clone ComfyUI
if not os.path.exists(COMFY_DIR):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    os.chdir(COMFY_DIR) and !git pull
os.chdir(COMFY_DIR)
!{pip} install -q -r requirements.txt
print('ComfyUI ready')

In [ ]:
%%time
# Clone custom nodes
os.chdir(f'{COMFY_DIR}/custom_nodes')
for repo in [
    ('Lightricks/ComfyUI-LTXVideo', 'ComfyUI-LTXVideo'),
    ('logtd/ComfyUI-LTXTricks', 'ComfyUI-LTXTricks'),
    ('city96/ComfyUI-GGUF', 'ComfyUI-GGUF'),
]:
    if not os.path.exists(repo[1]):
        !git clone https://github.com/{repo[0]}.git
        print(f'  {repo[1]} OK')
print('All custom nodes installed')

---
## 2. Download Models

In [ ]:
%%time
# == Download LTX-2.3 GGUF ==
model_dir = f'{COMFY_DIR}/models/unet'
os.makedirs(model_dir, exist_ok=True)
os.chdir(model_dir)

model_file = f'LTX-2.3-22B-distilled-1.1-{MODEL_QUANT}.gguf'
model_url = f'https://huggingface.co/QuantStack/LTX-2.3-GGUF/resolve/main/LTX-2.3-distilled-1.1/{model_file}'

if not os.path.exists(model_file):
    print('Downloading (~12 GB)...')
    !wget -c "{model_url}" -O "{model_file}" 2>&1
    print(f'Done: {os.path.getsize(model_file)/1e9:.1f} GB')
else:
    print(f'Already exists: {os.path.getsize(model_file)/1e9:.1f} GB')

In [ ]:
%%time
# == Download text encoder + VAE ==
clip_dir = f'{COMFY_DIR}/models/clip'
vae_dir = f'{COMFY_DIR}/models/vae'
os.makedirs(clip_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)

if not os.path.exists(f'{clip_dir}/gemma-2b.safetensors'):
    !wget -c "https://huggingface.co/Lightricks/LTX-2/resolve/main/text_encoder/model.safetensors" -O "{clip_dir}/gemma-2b.safetensors" 2>&1
if not os.path.exists(f'{vae_dir}/ltx-vae.safetensors'):
    !wget -c "https://huggingface.co/Lightricks/LTX-2/resolve/main/vae/vae.safetensors" -O "{vae_dir}/ltx-vae.safetensors" 2>&1
print('Text encoder + VAE ready')

---
## 3. Start ComfyUI + Tunnel Pinggy

In [ ]:
# == Start ComfyUI headless ==
os.chdir(COMFY_DIR)
log_file = f'{home_dir}/comfyui.log'
with open(log_file, 'w') as f:
    proc = subprocess.Popen(
        [python, 'main.py', '--headless', '--port', '8188', '--listen', '127.0.0.1',
         '--dont-print-server', '--highvram'],
        stdout=f, stderr=f
    )

time.sleep(10)
for i in range(30):
    try:
        r = requests.get('http://127.0.0.1:8188/object_info', timeout=2)
        if r.status_code == 200:
            print(f'ComfyUI ready! PID: {proc.pid}')
            break
    except:
        pass
    time.sleep(3)

In [ ]:
# == Tunnel Pinggy ==
TUNNEL_URL = None

def start_pinggy():
    global TUNNEL_URL
    cmd = 'ssh -p 443 -R0:localhost:8188 a.pinggy.io'
    proc = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    for line in proc.stdout:
        if 'https://' in line:
            url = line[line.find('https://'):].strip()
            if '.pinggy.link' in url or '.pinggy.io' in url:
                TUNNEL_URL = url
                with open(f'{home_dir}/tunnel_url.txt', 'w') as f:
                    f.write(url)
                print(f'\n\\nTUNNEL URL: {url}')
                break

t = threading.Thread(target=start_pinggy, daemon=True)
t.start()
time.sleep(15)

for i in range(20):
    if TUNNEL_URL: break
    try:
        with open(f'{home_dir}/tunnel_url.txt') as f:
            TUNNEL_URL = f.read().strip()
    except:
        pass
    time.sleep(5)

print(f'Tunnel URL: {TUNNEL_URL}')

---
## 4. Ghi URL lên GitHub Gist → Hermes biết

In [ ]:
# == Push URL lên Gist ==
def update_gist(url, status):
    data = {}
    try:
        r = requests.get(f'https://api.github.com/gists/{GIST_ID}', timeout=5)
        if r.status_code == 200:
            c = r.json()['files']['kaggle_backends.json']['content']
            data = json.loads(c) if c.strip() else {}
    except:
        pass

    data[BACKEND_NAME] = {
        'url': url,
        'status': status,
        'updated': datetime.now().isoformat(),
        'capabilities': ['t2v', 'i2v', 'v2v', 'motion_control'],
        'gpu': 't4',
    }
    r = requests.patch(f'https://api.github.com/gists/{GIST_ID}',
        json={'files': {'kaggle_backends.json': {'content': json.dumps(data, indent=2)}}},
        timeout=10)
    return r.status_code == 200

if TUNNEL_URL:
    ok = update_gist(TUNNEL_URL, 'online')
    print(f'Gist update: {ok}')
    print()
    print(f'API: POST {TUNNEL_URL}/prompt')
else:
    print('No tunnel URL')

---
## 5. Keep Alive — Heartbeat m?i 5 phút

In [ ]:
# == Heartbeat loop ==
print(f'Backend: {BACKEND_NAME}')
print(f'Tunnel: {TUNNEL_URL}')
print('Waiting for jobs from Hermes...')

try:
    while True:
        time.sleep(300)  # 5 phút
        update_gist(TUNNEL_URL, 'online')
        print(f'Heartbeat: {datetime.now().isoformat()}')
except KeyboardInterrupt:
    update_gist(TUNNEL_URL, 'offline')
    print('Stopped')